# 03 - EDA Pix

Este notebook realiza análise exploratória didática a partir da camada Silver.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pyspark.sql import functions as F

from src.config import FIGURES_DIR, PIX_CLEAN_DIR, PIX_EDA_SUMMARY_DIR, create_project_directories
from src.data_quality import ensure_not_empty
from src.spark_session import get_spark_session

create_project_directories(False)
spark = get_spark_session("03-eda-pix")

In [ ]:
silver_df = spark.read.parquet(str(PIX_CLEAN_DIR))
ensure_not_empty(silver_df, "Silver Pix clean")
silver_df.printSchema()
print(f"Registros: {silver_df.count()}")
silver_df.show(5, truncate=False)

In [ ]:
summary_rows = []
summary_rows.append(("total_registros", float(silver_df.count())))
summary_rows.append(("periodo_minimo", float(silver_df.agg(F.min("ano_mes")).first()[0])))
summary_rows.append(("periodo_maximo", float(silver_df.agg(F.max("ano_mes")).first()[0])))
summary_rows.append(("valor_total", float(silver_df.agg(F.sum("valor_total")).first()[0])))
summary_rows.append(("quantidade_transacoes", float(silver_df.agg(F.sum("quantidade_transacoes")).first()[0])))

null_rows = []
for column in ["ano_mes", "quantidade_transacoes", "valor_total"]:
    null_rows.append((f"nulos_{column}", float(silver_df.filter(F.col(column).isNull()).count())))
summary_rows.extend(null_rows)

duplicates = silver_df.groupBy(silver_df.columns).count().filter(F.col("count") > 1).count()
summary_rows.append(("linhas_duplicadas", float(duplicates)))

eda_summary_df = spark.createDataFrame(summary_rows, ["metrica", "valor"])
eda_summary_df.write.mode("overwrite").parquet(str(PIX_EDA_SUMMARY_DIR))
eda_summary_df.show(truncate=False)

In [ ]:
monthly_pd = (
    silver_df.groupBy("ano_mes")
    .agg(
        F.sum("quantidade_transacoes").alias("quantidade_transacoes"),
        F.sum("valor_total").alias("valor_total"),
    )
    .withColumn("ticket_medio", F.when(F.col("quantidade_transacoes") > 0, F.col("valor_total") / F.col("quantidade_transacoes")))
    .orderBy("ano_mes")
    .toPandas()
)
print(monthly_pd.describe().to_string())
print("Pico valor:", monthly_pd.loc[monthly_pd["valor_total"].idxmax()].to_dict())
print("Vale valor:", monthly_pd.loc[monthly_pd["valor_total"].idxmin()].to_dict())

In [ ]:
plt.figure(figsize=(12, 6))
plt.hist(monthly_pd["valor_total"] / 1_000_000_000, bins=10, color="#1f77b4", alpha=0.85)
plt.title("Distribuição mensal do valor financeiro Pix")
plt.xlabel("Valor financeiro (R$ bilhões)")
plt.ylabel("Frequência")
plt.grid(True, alpha=0.3)
plt.figtext(0.01, 0.01, "Fonte: dados públicos do Banco Central do Brasil", fontsize=9)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.savefig(FIGURES_DIR / "08_pix_eda_distribution_value.png", dpi=160)
plt.close()

plt.figure(figsize=(12, 6))
plt.hist(monthly_pd["quantidade_transacoes"] / 1_000_000_000, bins=10, color="#2ca02c", alpha=0.85)
plt.title("Distribuição mensal da quantidade de transações Pix")
plt.xlabel("Quantidade de transações (bilhões)")
plt.ylabel("Frequência")
plt.grid(True, alpha=0.3)
plt.figtext(0.01, 0.01, "Fonte: dados públicos do Banco Central do Brasil", fontsize=9)
plt.tight_layout(rect=(0, 0.04, 1, 1))
plt.savefig(FIGURES_DIR / "09_pix_eda_distribution_transactions.png", dpi=160)
plt.close()

## Achados

A EDA consolida contagens, período, nulos, duplicidades e distribuições mensais. Picos e vales indicam meses que merecem investigação adicional, sem implicar causalidade.

In [ ]:
spark.stop()